# card_stack

> Card stack operations for the alignment column: navigation, viewport update, width save

In [ ]:
#| default_exp routes.card_stack

In [ ]:
#| export
from typing import Any, Dict, List, Tuple, Callable

from fasthtml.common import APIRouter

from cjm_fasthtml_interactions.core.state_store import get_session_id
from cjm_fasthtml_card_stack.core.models import CardStackState
from cjm_fasthtml_card_stack.core.constants import DEFAULT_CARD_WIDTH
from cjm_fasthtml_card_stack.routes.handlers import (
    build_nav_response, card_stack_navigate,
    card_stack_update_viewport, card_stack_save_width,
)

from cjm_transcript_vad_align.components.card_stack_config import (
    ALIGN_CS_CONFIG, ALIGN_CS_IDS, ALIGN_CS_BTN_IDS,
)
from cjm_transcript_vad_align.components.vad_card import (
    create_vad_card_renderer,
)
from cjm_transcript_vad_align.models import AlignmentUrls
from cjm_transcript_vad_align.routes.core import (
    WorkflowStateStore, _load_alignment_context, _update_alignment_state,
    _build_card_stack_state, _to_vad_chunks,
)

## Navigation Response Builder

In [ ]:
#| export
def _build_nav_response(
    chunk_dicts:List[Dict[str, Any]],  # Serialized VAD chunks
    state:CardStackState,  # Current card stack state
    urls:AlignmentUrls,  # URL bundle
) -> Tuple:  # OOB response elements (slots + progress + focus)
    """Build OOB response for navigation changes."""
    chunks = _to_vad_chunks(chunk_dicts)
    return build_nav_response(
        card_items=chunks,
        state=state,
        config=ALIGN_CS_CONFIG,
        ids=ALIGN_CS_IDS,
        urls=urls.card_stack,
        render_card=create_vad_card_renderer(),
        progress_label="VAD Chunk",
        form_input_name="chunk_index",
    )

## Route Handlers

In [ ]:
#| export
def _handle_align_navigate(
    state_store:WorkflowStateStore,  # The workflow state store
    workflow_id:str,  # The workflow identifier
    sess:Any,  # FastHTML session object
    direction:str,  # Navigation direction: up/down/first/last/page_up/page_down
    urls:AlignmentUrls,  # URL bundle
) -> Tuple:  # OOB response elements (slots + progress + focus)
    """Navigate the alignment card stack."""
    session_id = get_session_id(sess)
    ctx = _load_alignment_context(state_store, workflow_id, session_id)
    chunks = _to_vad_chunks(ctx.chunk_dicts)

    state = _build_card_stack_state(ctx)
    renderer = create_vad_card_renderer()

    result = card_stack_navigate(
        direction=direction,
        card_items=chunks,
        state=state,
        config=ALIGN_CS_CONFIG,
        ids=ALIGN_CS_IDS,
        urls=urls.card_stack,
        render_card=renderer,
        progress_label="VAD Chunk",
        form_input_name="chunk_index",
    )

    _update_alignment_state(state_store, workflow_id, session_id, focused_chunk_index=state.focused_index)
    return result

In [ ]:
#| export
async def _handle_align_update_viewport(
    state_store:WorkflowStateStore,  # The workflow state store
    workflow_id:str,  # The workflow identifier
    request:Any,  # FastHTML request object
    sess:Any,  # FastHTML session object
    visible_count:int,  # New visible card count
    urls:AlignmentUrls,  # URL bundle
) -> Tuple:  # OOB section elements for viewport update
    """Update viewport with new card count via OOB section swaps."""
    session_id = get_session_id(sess)
    ctx = _load_alignment_context(state_store, workflow_id, session_id)
    chunks = _to_vad_chunks(ctx.chunk_dicts)

    state = _build_card_stack_state(ctx)
    renderer = create_vad_card_renderer()

    result = card_stack_update_viewport(
        visible_count=visible_count,
        card_items=chunks,
        state=state,
        config=ALIGN_CS_CONFIG,
        ids=ALIGN_CS_IDS,
        urls=urls.card_stack,
        render_card=renderer,
    )

    # Read is_auto from form data (passed by client JS)
    form_data = await request.form()
    is_auto_str = form_data.get("is_auto", "false")
    is_auto_mode = is_auto_str.lower() == "true"

    _update_alignment_state(
        state_store, workflow_id, session_id,
        visible_count=state.visible_count,
        is_auto_mode=is_auto_mode,
    )
    return result

In [ ]:
#| export
def _handle_align_save_width(
    state_store:WorkflowStateStore,  # The workflow state store
    workflow_id:str,  # The workflow identifier
    sess:Any,  # FastHTML session object
    card_width:int,  # Card stack width in rem to save
) -> None:  # No response body (swap=none on client)
    """Save card stack width to server state with config bounds validation."""
    session_id = get_session_id(sess)
    ctx = _load_alignment_context(state_store, workflow_id, session_id)
    state = _build_card_stack_state(ctx)
    card_stack_save_width(state, card_width, ALIGN_CS_CONFIG)
    _update_alignment_state(state_store, workflow_id, session_id, card_width=state.card_width)

## Router Initialization

Creates the card stack router with navigation and viewport routes.

In [ ]:
#| export
def init_card_stack_router(
    state_store:WorkflowStateStore,  # The workflow state store
    workflow_id:str,  # The workflow identifier
    prefix:str,  # Route prefix (e.g., "/workflow/align/card_stack")
    urls:AlignmentUrls,  # URL bundle (populated after routes defined)
) -> Tuple[APIRouter, Dict[str, Callable]]:  # (router, route_dict)
    """Initialize card stack routes for alignment."""
    router = APIRouter(prefix=prefix)

    # -------------------------------------------------------------------------
    # Navigation
    # -------------------------------------------------------------------------

    @router
    def nav_up(request, sess):
        """Navigate to previous VAD chunk."""
        return _handle_align_navigate(state_store, workflow_id, sess, direction="up", urls=urls)

    @router
    def nav_down(request, sess):
        """Navigate to next VAD chunk."""
        return _handle_align_navigate(state_store, workflow_id, sess, direction="down", urls=urls)

    @router
    def nav_first(request, sess):
        """Navigate to first VAD chunk."""
        return _handle_align_navigate(state_store, workflow_id, sess, direction="first", urls=urls)

    @router
    def nav_last(request, sess):
        """Navigate to last VAD chunk."""
        return _handle_align_navigate(state_store, workflow_id, sess, direction="last", urls=urls)

    @router
    def nav_page_up(request, sess):
        """Navigate up by page in alignment."""
        return _handle_align_navigate(state_store, workflow_id, sess, direction="page_up", urls=urls)

    @router
    def nav_page_down(request, sess):
        """Navigate down by page in alignment."""
        return _handle_align_navigate(state_store, workflow_id, sess, direction="page_down", urls=urls)

    # -------------------------------------------------------------------------
    # Viewport and Width
    # -------------------------------------------------------------------------

    @router
    async def update_viewport(request, sess, visible_count: int):
        """Update alignment viewport with new card count."""
        return await _handle_align_update_viewport(
            state_store, workflow_id, request, sess, visible_count, urls=urls,
        )

    @router
    def save_width(request, sess, card_width: int):
        """Save alignment card stack width to server state."""
        return _handle_align_save_width(state_store, workflow_id, sess, card_width)

    # -------------------------------------------------------------------------
    # Route Dict
    # -------------------------------------------------------------------------

    routes = {
        "nav_up": nav_up,
        "nav_down": nav_down,
        "nav_first": nav_first,
        "nav_last": nav_last,
        "nav_page_up": nav_page_up,
        "nav_page_down": nav_page_down,
        "update_viewport": update_viewport,
        "save_width": save_width,
    }

    return router, routes

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()